#### Import Necessary Libraries

In [1]:
#Data manipulation
import pandas as pd
import numpy as np

# Plotting
import matplotlib.pyplot as plt

# Preprocessing
from sklearn.preprocessing import MinMaxScaler
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.model_selection import (
train_test_split,
RandomizedSearchCV,
TimeSeriesSplit,
cross_val_score
)

# K-mean clustering
from sklearn.cluster import KMeans

# Classifier
from sklearn.ensemble import RandomForestClassifier

# Metrics
from sklearn.metrics import (precision_recall_curve,
roc_curve,
RocCurveDisplay,
ConfusionMatrixDisplay
)

from sklearn.metrics import (accuracy_score,
f1_score,
recall_score,
precision_score,
roc_auc_score,
auc
)

from sklearn.metrics import (classification_report,
confusion_matrix
)

# Import tensorflow and keras
import tensorflow
import keras 

# Import pandas-ta
import pandas_ta as ta

#### Retrieving the Historical Price

In [2]:
files = ["CMA.csv", "HML.csv", "MKT.csv", "RMW.csv", "SMB.csv", "UMD.csv"]
tickers = ["cma", "hml", "mkt", "rmw", "smb", "umd",]

data = pd.DataFrame()

# Load files, extract the 'Close' column, and concatenate DataFrames
for i, file in enumerate(files):
    df = pd.read_csv(file, index_col=0, parse_dates=True, usecols=['date', 'close'])
    df.columns = [f"Close_{tickers[i]}"]  # Rename 'Close' column with ticker suffix
    data = pd.concat([data, df], axis=1)

# Also save each original ticker data into a single dataframe for later calculation of technical indicators
for ticker in tickers:
    globals()[ticker] = pd.read_csv(ticker + ".csv", index_col=0, parse_dates=True)

# Save the concatenated DataFrame to a new CSV file
data.to_csv("concatenated_closes.csv")

data


,Close_cma,Close_hml,Close_mkt,Close_rmw,Close_smb,Close_umd
date,,,,,,
2/1/2014,100.557344,101.654116,94.677887,100.240243,101.235292,98.681401
3/1/2014,100.452750,103.314478,94.177525,99.544800,100.475639,98.564815
6/1/2014,100.386385,102.445666,94.693658,98.665899,98.574632,99.743182
7/1/2014,99.739388,100.809123,97.162546,99.425262,98.126280,100.336994
8/1/2014,100.057959,100.218158,96.824794,99.817369,97.780785,100.198034
...,...,...,...,...,...,...
24/4/2024,163.279450,137.004451,127.042559,105.443845,116.124809,191.263376
25/4/2024,162.954590,136.450907,127.374724,105.673171,116.582596,191.810751
26/4/2024,161.715370,137.514225,126.979863,106.109936,116.859371,191.888061


## Exploratory Data Analysis

In [3]:
data.describe().T

,count,mean,std,min,25%,50%,75%,max
Close_cma,2507.0,121.447539,16.351098,99.142622,107.595953,117.245210,129.907561,166.667970
Close_hml,2507.0,130.463038,17.471080,99.472920,119.446821,131.180002,140.382946,170.913791
Close_mkt,2507.0,131.681843,14.473155,89.455829,120.994125,132.149616,143.955931,155.459147
Close_rmw,2507.0,104.424673,7.689591,84.335946,99.687239,103.522896,108.547883,133.273850
Close_smb,2507.0,108.093572,14.970156,73.234252,95.702319,114.557540,119.913614,130.826035
Close_umd,2507.0,133.747526,23.839062,94.153230,110.983528,138.850625,146.072182,197.827201


## Data Cleaning

In [4]:
# check for missing values
data.isnull().sum()

Close_cma    0
Close_hml    0
Close_mkt    0
Close_rmw    0
Close_smb    0
Close_umd    0
dtype: int64

## Data Transformation

### Feature Engineering


In [5]:
files = ["CMA.csv", "HML.csv", "MKT.csv", "RMW.csv", "SMB.csv", "UMD.csv"]
tickers = ["cma", "hml", "mkt", "rmw", "smb", "umd"]

ret = pd.DataFrame()

# Load files, extract the 'RET' column, and concatenate DataFrames
for i, file in enumerate(files):
    df = pd.read_csv(file, index_col=0, parse_dates=True, usecols=['date', 'return'])
    df.columns = [f"RET_{tickers[i]}"]  # Rename 'RET' column with ticker suffix
    ret = pd.concat([ret, df], axis=1)

# Save the concatenated DataFrame to a new CSV file if needed
ret.to_csv("concatenated_returns.csv")

ret

,RET_cma,RET_hml,RET_mkt,RET_rmw,RET_smb,RET_umd
date,,,,,,
2/1/2014,0.005573,0.016541,-0.053221,0.002402,0.012353,-0.013186
3/1/2014,-0.001040,0.016333,-0.005285,-0.006938,-0.007504,-0.001181
6/1/2014,-0.000661,-0.008409,0.005480,-0.008829,-0.018920,0.011955
7/1/2014,-0.006445,-0.015975,0.026072,0.007696,-0.004548,0.005953
8/1/2014,0.003194,-0.005862,-0.003476,0.003944,-0.003521,-0.001385
...,...,...,...,...,...,...
24/4/2024,-0.006187,-0.010302,0.003068,0.003958,-0.005578,0.005125
25/4/2024,-0.001990,-0.004040,0.002615,0.002175,0.003942,0.002862
26/4/2024,-0.007605,0.007793,-0.003100,0.004133,0.002374,0.000403


In [6]:
# Define the SMA periods
sma_periods = [5, 10, 20, 50, 100, 200]

# Loop through each ticker and calculate SMA for each period
for ticker in tickers:
    # Identify the column that contains the close prices (assuming the column name is 'close' followed by the ticker)
    close_col = f"Close_{ticker}"
    
    # Check if the column exists in the data DataFrame
    if close_col in data.columns:
        for period in sma_periods:
            # Calculate SMA
            sma_col = f"SMA_{period}_{ticker}"
            sma_series = ta.sma(data[close_col], length=period)
            
            # Add SMA to the DataFrame with appropriate name
            data[sma_col] = sma_series
    else:
        print(f"Column {close_col} does not exist in the DataFrame.")

print(data)

            Close_cma   Close_hml   Close_mkt   Close_rmw   Close_smb  \
date                                                                    
2/1/2014   100.557344  101.654116   94.677887  100.240243  101.235292   
3/1/2014   100.452750  103.314478   94.177525   99.544800  100.475639   
6/1/2014   100.386385  102.445666   94.693658   98.665899   98.574632   
7/1/2014    99.739388  100.809123   97.162546   99.425262   98.126280   
8/1/2014   100.057959  100.218158   96.824794   99.817369   97.780785   
...               ...         ...         ...         ...         ...   
24/4/2024  163.279450  137.004451  127.042559  105.443845  116.124809   
25/4/2024  162.954590  136.450907  127.374724  105.673171  116.582596   
26/4/2024  161.715370  137.514225  126.979863  106.109936  116.859371   
29/4/2024  163.066388  138.917666  127.191510  105.437125  117.486626   
30/4/2024  163.662501  138.063658  127.798171  105.398411  117.506506   

            Close_umd   SMA_5_cma  SMA_10_cma  SMA

In [7]:
# Define the EMA periods
ema_periods = [5, 8, 10, 12, 25, 26, 50, 200]

# Loop through each ticker and calculate EMA for each period
for ticker in tickers:
    # Identify the column that contains the close prices (assuming the column name is 'close' followed by the ticker)
    close_col = f"Close_{ticker}"
    
    # Check if the column exists in the data DataFrame
    if close_col in data.columns:
        for period in ema_periods:
            # Calculate EMA
            ema_col = f"EMA_{period}_{ticker}"
            ema_series = ta.ema(data[close_col], length=period)
            
            # Add EMA to the DataFrame with appropriate name
            data[ema_col] = ema_series
    else:
        print(f"Column {close_col} does not exist in the DataFrame.")

print(data) 

            Close_cma   Close_hml   Close_mkt   Close_rmw   Close_smb  \
date                                                                    
2/1/2014   100.557344  101.654116   94.677887  100.240243  101.235292   
3/1/2014   100.452750  103.314478   94.177525   99.544800  100.475639   
6/1/2014   100.386385  102.445666   94.693658   98.665899   98.574632   
7/1/2014    99.739388  100.809123   97.162546   99.425262   98.126280   
8/1/2014   100.057959  100.218158   96.824794   99.817369   97.780785   
...               ...         ...         ...         ...         ...   
24/4/2024  163.279450  137.004451  127.042559  105.443845  116.124809   
25/4/2024  162.954590  136.450907  127.374724  105.673171  116.582596   
26/4/2024  161.715370  137.514225  126.979863  106.109936  116.859371   
29/4/2024  163.066388  138.917666  127.191510  105.437125  117.486626   
30/4/2024  163.662501  138.063658  127.798171  105.398411  117.506506   

            Close_umd   SMA_5_cma  SMA_10_cma  SMA

In [8]:
# Define the RSI periods
rsi_periods = [5, 9, 14, 30]

# Loop through each ticker and calculate RSI for each period
for ticker in tickers:
    # Identify the column that contains the close prices (assuming the column name is 'close' followed by the ticker)
    close_col = f"Close_{ticker}"
    
    # Check if the column exists in the data DataFrame
    if close_col in data.columns:
        for period in rsi_periods:
            # Calculate RSI
            rsi_col = f"RSI_{period}_{ticker}"
            rsi_series = ta.rsi(data[close_col], length=period)
            
            # Add RSI to the DataFrame with appropriate name
            data[rsi_col] = rsi_series
    else:
        print(f"Column {close_col} does not exist in the DataFrame.")

print(data)

            Close_cma   Close_hml   Close_mkt   Close_rmw   Close_smb  \
date                                                                    
2/1/2014   100.557344  101.654116   94.677887  100.240243  101.235292   
3/1/2014   100.452750  103.314478   94.177525   99.544800  100.475639   
6/1/2014   100.386385  102.445666   94.693658   98.665899   98.574632   
7/1/2014    99.739388  100.809123   97.162546   99.425262   98.126280   
8/1/2014   100.057959  100.218158   96.824794   99.817369   97.780785   
...               ...         ...         ...         ...         ...   
24/4/2024  163.279450  137.004451  127.042559  105.443845  116.124809   
25/4/2024  162.954590  136.450907  127.374724  105.673171  116.582596   
26/4/2024  161.715370  137.514225  126.979863  106.109936  116.859371   
29/4/2024  163.066388  138.917666  127.191510  105.437125  117.486626   
30/4/2024  163.662501  138.063658  127.798171  105.398411  117.506506   

            Close_umd   SMA_5_cma  SMA_10_cma  SMA

C:\Users\tonfo\AppData\Local\Temp\ipykernel_35004\957566562.py:17: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  data[rsi_col] = rsi_series
C:\Users\tonfo\AppData\Local\Temp\ipykernel_35004\957566562.py:17: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  data[rsi_col] = rsi_series
C:\Users\tonfo\AppData\Local\Temp\ipykernel_35004\957566562.py:17: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once usi

In [9]:
# Define the Bollinger Bands parameters
bb_params = [
    (10, 1.5),
    (20, 2),
    (50, 2.5)
]

# Loop through each ticker and calculate Bollinger Bands for each parameter set
for ticker in tickers:
    # Identify the column that contains the close prices (assuming the column name is 'close' followed by the ticker)
    close_col = f"Close_{ticker}"
    
    # Check if the column exists in the data DataFrame
    if close_col in data.columns:
        for period, std_dev in bb_params:
            # Calculate Bollinger Bands
            bb = ta.bbands(data[close_col], length=period, std=std_dev)
            
            # Extract the Bollinger Bands components from the returned DataFrame
            bb_upper_col = bb.columns[0]
            bb_middle_col = bb.columns[1]
            bb_lower_col = bb.columns[2]
            
            bb_upper = bb[bb_upper_col]
            bb_middle = bb[bb_middle_col]
            bb_lower = bb[bb_lower_col]
            
            # Add Bollinger Bands components to the DataFrame with appropriate names
            data[f"BB_upper_{period}_{std_dev}_{ticker}"] = bb_upper
            data[f"BB_middle_{period}_{std_dev}_{ticker}"] = bb_middle
            data[f"BB_lower_{period}_{std_dev}_{ticker}"] = bb_lower
    else:
        print(f"Column {close_col} does not exist in the DataFrame.")

print(data)

            Close_cma   Close_hml   Close_mkt   Close_rmw   Close_smb  \
date                                                                    
2/1/2014   100.557344  101.654116   94.677887  100.240243  101.235292   
3/1/2014   100.452750  103.314478   94.177525   99.544800  100.475639   
6/1/2014   100.386385  102.445666   94.693658   98.665899   98.574632   
7/1/2014    99.739388  100.809123   97.162546   99.425262   98.126280   
8/1/2014   100.057959  100.218158   96.824794   99.817369   97.780785   
...               ...         ...         ...         ...         ...   
24/4/2024  163.279450  137.004451  127.042559  105.443845  116.124809   
25/4/2024  162.954590  136.450907  127.374724  105.673171  116.582596   
26/4/2024  161.715370  137.514225  126.979863  106.109936  116.859371   
29/4/2024  163.066388  138.917666  127.191510  105.437125  117.486626   
30/4/2024  163.662501  138.063658  127.798171  105.398411  117.506506   

            Close_umd   SMA_5_cma  SMA_10_cma  SMA

C:\Users\tonfo\AppData\Local\Temp\ipykernel_35004\3031231226.py:29: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  data[f"BB_upper_{period}_{std_dev}_{ticker}"] = bb_upper
C:\Users\tonfo\AppData\Local\Temp\ipykernel_35004\3031231226.py:30: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  data[f"BB_middle_{period}_{std_dev}_{ticker}"] = bb_middle
C:\Users\tonfo\AppData\Local\Temp\ipykernel_35004\3031231226.py:31: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, whic

In [10]:
# Build the feature matrix
feat = pd.concat([ret, data], axis=1) # Feature matrix

# Save the matrix
feat.to_csv('feat.csv')

c:\Users\tonfo\AppData\Local\Programs\Python\Python311\Lib\site-packages\pandas\core\indexes\base.py:7834: RuntimeWarning: invalid value encountered in cast
  values = values.astype(str)


### Label Definition
For CMA 

In [11]:
#return 6 m from set index
df = pd.read_csv('SETReturn.csv')
set_6m = df['return 6m']

#return 6 m from CMA
df = pd.read_csv('CMA.csv')
cma_6m = df['return 6m']
cma_6m = df['return 6m'].rename(columns={'return 6m': 'cma_6m'})

# Build the feature matrix
cma_m = pd.concat([feat, set_6m, cma_6m ])

#save to csv
cma_m.to_csv('cma_model')

KeyError: 'return 6m'

In [ ]:
df = pd.read_csv('cma_model.csv')

# Define target
df['Target'] = np.where(df['cma 6 m'] > np.log(1.05) * df['return 6 m'], 1, 0)

# Drop the last row where there is no label
df = df.iloc[:-1]

# Drop the rows with NaN value
df.dropna(inplace=True)

X = df.drop(['Target'],axis=1)
y = df['Target']


FileNotFoundError: [Errno 2] No such file or directory: 'cma_model.csv'

In [ ]:
#check data
X.to_csv('X_cma')
y.to_csv('y_cma')

### Data Splitting

In [ ]:
X_train,X_test = X[0:600],X[779:780]
y_train, y_test= y[0:600],y[779:780]

# Output the training and test data size
print(f"Train and Test Size {len(X_train)}, {len(X_test)}")

### Data Scaling

In [ ]:
# Use StandardScaler
scaler = StandardScaler()
X_train_scaled = pd.DataFrame(scaler.fit_transform(X_train), index=X_train.index, columns=X_train.columns)

# Apply the scaler to the test set
X_test_scaled = pd.DataFrame(scaler.transform(X_test), index=X_test.index, columns=X_test.columns)

X_train_scaled.describe()

In [ ]:
# Store the scaled data in the original variables
X_train = X_train_scaled
X_test = X_test_scaled

### K-means Clustering

In [ ]:
# K-mean clustering
inertia = []

K = range(1, 50)

for k in K:
    kmeans = KMeans(n_clusters=k, random_state=999)
    kmeans.fit(X_train.T)
    inertia.append(kmeans.inertia_)